# Ejecución presupuestal — Gobierno Regional de Piura (2026)

Replica el cuadro de PIA, PIM, Certificación, Compromiso Anual, Devengado, Girado y % de Avance
por Unidad Ejecutora, a partir del archivo `2026-Gasto-Mensual.csv` descargado del
[Portal de Datos Abiertos del MEF](https://datosabiertos.mef.gob.pe/dataset/presupuesto-y-ejecucion-de-gasto/resource/d45f660d-6d14-438e-9d91-300084c9b85f).

**Cómo usar este notebook:**
1. Corre las celdas en orden (Shift+Enter en cada una).
2. En la celda de **Configuración**, confirma que `RUTA_ARCHIVO` apunte a tu CSV.
3. Corre la celda de **Ejecución** al final. Si el filtro de Piura no encuentra filas,
   revisa el archivo `diagnostico_piura.txt` que se genera junto a tu CSV.


In [ ]:
import os
import pandas as pd


In [ ]:
# ---------------------------------------------------------------------------
# CONFIGURACIÓN
# ---------------------------------------------------------------------------
RUTA_ARCHIVO = r"C:\Users\Comercial-LT03\OneDrive\Escritorio\TAREA_2\2026-Gasto-Mensual.csv"

# Si el Excel/CSV tiene varias hojas o filas de encabezado antes de la tabla,
# ajusta estos dos parámetros (0 = primera hoja / sin filas extra arriba).
NOMBRE_HOJA = 0
FILAS_A_SALTAR = 0

# Filtro geográfico/institucional: Gobierno Regional de Piura
# (confirmados en el diagnóstico; vuelve a verificarlos si cambias de archivo)
DEPARTAMENTO = "PIURA"
NIVEL_GOBIERNO = "GOBIERNOS REGIONALES"
PLIEGO_CONTIENE = "GOBIERNO REGIONAL DEL DEPARTAMENTO DE PIURA"


In [ ]:
# ---------------------------------------------------------------------------
# LECTURA DEL ARCHIVO LOCAL
# ---------------------------------------------------------------------------
def leer_csv_autodetectando_separador(ruta):
    """Prueba encoding y separador más comunes (engine C, rápido) y se queda
    con la combinación que produce más columnas."""
    encodings = ["utf-8-sig", "latin-1"]
    separadores = [",", ";", "\t", "|"]

    mejor_config = None
    mejor_num_columnas = 1

    for enc in encodings:
        for sep in separadores:
            try:
                candidato = pd.read_csv(
                    ruta, sep=sep, quotechar='"', encoding=enc,
                    skiprows=FILAS_A_SALTAR, nrows=200,  # muestra rápida, engine C por defecto
                )
            except Exception:
                continue
            if candidato.shape[1] > mejor_num_columnas:
                mejor_num_columnas = candidato.shape[1]
                mejor_config = (enc, sep)

    if mejor_config is None:
        raise ValueError(
            "No se pudo detectar automáticamente el separador/encoding del CSV. "
            "Abre el archivo en el Bloc de notas y revisa manualmente qué carácter "
            "separa las columnas (coma, punto y coma, tabulación)."
        )

    enc, sep = mejor_config
    print(f"[info] Separador detectado: {sep!r} | Encoding: {enc}")
    print("[info] Cargando archivo completo (puede tardar si es grande)...")
    try:
        return pd.read_csv(
            ruta, sep=sep, quotechar='"', encoding=enc,
            skiprows=FILAS_A_SALTAR, low_memory=False,  # engine C: rápido
        )
    except Exception:
        # Fallback si el engine C no puede con líneas irregulares
        return pd.read_csv(
            ruta, sep=sep, quotechar='"', encoding=enc,
            skiprows=FILAS_A_SALTAR, engine="python", on_bad_lines="warn",
        )


def leer_archivo(ruta):
    if not os.path.exists(ruta):
        raise FileNotFoundError(f"No se encontró el archivo en: {ruta}")

    extension = os.path.splitext(ruta)[1].lower()

    if extension in (".xlsx", ".xls"):
        df = pd.read_excel(ruta, sheet_name=NOMBRE_HOJA, skiprows=FILAS_A_SALTAR)
    elif extension == ".csv":
        df = leer_csv_autodetectando_separador(ruta)
    else:
        raise ValueError(f"Extensión no soportada: {extension} (usa .xlsx, .xls o .csv)")

    if df.shape[1] <= 1:
        raise ValueError(
            "Se leyó el archivo pero solo se detectó 1 columna: revisa si hay filas "
            "de encabezado extra arriba de la tabla (ajusta FILAS_A_SALTAR) o si el "
            "separador del CSV es distinto (coma vs punto y coma)."
        )

    return df


In [ ]:
# ---------------------------------------------------------------------------
# DIAGNÓSTICO — confirmar nombres de columnas reales del archivo
# ---------------------------------------------------------------------------
def mostrar_columnas(df):
    print("Columnas disponibles en el archivo:")
    for c in df.columns:
        print(" -", c)


In [ ]:
# ---------------------------------------------------------------------------
# FILTRADO Y AGRUPACIÓN
# Estructura real confirmada: MES_EJE == 0 trae el presupuesto anual
# (PIA, PIM, Compromiso Anual); MES_EJE 1..12 trae la ejecución mensual
# (Certificación, Compromiso, Devengado, Girado) que hay que sumar.
# ---------------------------------------------------------------------------
def construir_tabla(df):
    col_pliego = "PLIEGO_NOMBRE"
    col_depto = "DEPARTAMENTO_EJECUTORA_NOMBRE"
    col_nivel = "NIVEL_GOBIERNO_NOMBRE"
    col_ejecutora = "EJECUTORA_NOMBRE"
    col_mes = "MES_EJE"

    col_pia = "MONTO_PIA"
    col_pim = "MONTO_PIM"
    col_comp_anual = "MONTO_COMPROMETIDO_ANUAL"
    # columna de certificación: puede llamarse MONTO_CERTIFICADO o MONTO_CERTIFICADO_ANUAL según el archivo
    col_cert = "MONTO_CERTIFICADO_ANUAL" if "MONTO_CERTIFICADO_ANUAL" in df.columns else "MONTO_CERTIFICADO"
    col_comp_mensual = "MONTO_COMPROMETIDO"
    col_deveng = "MONTO_DEVENGADO_ANUAL" if "MONTO_DEVENGADO_ANUAL" in df.columns else "MONTO_DEVENGADO"
    col_girado = "MONTO_GIRADO_ANUAL" if "MONTO_GIRADO_ANUAL" in df.columns else "MONTO_GIRADO"

    montos_anuales = [c for c in [col_pia, col_pim, col_comp_anual] if c in df.columns]
    montos_ejecucion = [c for c in [col_cert, col_comp_mensual, col_deveng, col_girado] if c in df.columns]

    filtro = (
        df[col_depto].astype(str).str.upper().str.contains(DEPARTAMENTO, na=False)
        & df[col_nivel].astype(str).str.upper().str.contains(NIVEL_GOBIERNO.upper(), na=False)
        & df[col_pliego].astype(str).str.upper().str.contains(PLIEGO_CONTIENE, na=False)
    )
    df_piura = df.loc[filtro].copy()

    if df_piura.empty:
        diag_path = os.path.join(os.path.dirname(os.path.abspath(RUTA_ARCHIVO)), "diagnostico_piura.txt")
        with open(diag_path, "w", encoding="utf-8") as f:
            f.write("Valores únicos de NIVEL_GOBIERNO_NOMBRE:\n")
            for v in df[col_nivel].dropna().unique():
                f.write(f"  {v!r}\n")
            f.write("\nValores de PLIEGO_NOMBRE que contienen \'PIURA\':\n")
            for v in df.loc[df[col_pliego].astype(str).str.upper().str.contains("PIURA", na=False), col_pliego].unique():
                f.write(f"  {v!r}\n")
        print("\n[diagnóstico] El filtro no encontró filas.")
        print(f"[diagnóstico] Revisa el archivo generado en: {diag_path}")
        raise ValueError(
            "Revisa el archivo de diagnóstico generado y ajusta las constantes "
            "NIVEL_GOBIERNO / PLIEGO_CONTIENE al texto exacto que aparece ahí."
        )

    # Asegurar tipos numéricos
    for c in set(montos_anuales + montos_ejecucion):
        if df_piura[c].dtype == object:
            df_piura[c] = pd.to_numeric(
                df_piura[c].astype(str).str.replace(".", "", regex=False).str.replace(",", ".", regex=False),
                errors="coerce",
            )

    if col_mes in df_piura.columns:
        df_anual = df_piura[df_piura[col_mes] == 0]         # presupuesto (PIA/PIM/Compromiso Anual)
        df_ejecucion = df_piura[df_piura[col_mes] > 0]       # ejecución mensual acumulada
        print(f"[info] Filas Piura con MES_EJE==0 (presupuesto): {len(df_anual)}")
        print(f"[info] Filas Piura con MES_EJE>0 (ejecución): {len(df_ejecucion)}")
        print(f"[info] Meses de ejecución presentes: {sorted(df_ejecucion[col_mes].unique())}")
    else:
        df_anual = df_piura
        df_ejecucion = df_piura

    tabla_anual = df_anual.groupby(col_ejecutora)[montos_anuales].sum().reset_index()
    tabla_ejec = df_ejecucion.groupby(col_ejecutora)[montos_ejecucion].sum().reset_index()
    tabla = tabla_anual.merge(tabla_ejec, on=col_ejecutora, how="outer")

    renombres = {col_ejecutora: "Unidad Ejecutora"}
    if col_pia in tabla.columns: renombres[col_pia] = "PIA"
    if col_pim in tabla.columns: renombres[col_pim] = "PIM"
    if col_comp_anual in tabla.columns: renombres[col_comp_anual] = "Compromiso Anual"
    if col_cert in tabla.columns: renombres[col_cert] = "Certificación"
    if col_comp_mensual in tabla.columns: renombres[col_comp_mensual] = "Ejecución Comprometida"
    if col_deveng in tabla.columns: renombres[col_deveng] = "Devengado"
    if col_girado in tabla.columns: renombres[col_girado] = "Girado"
    tabla = tabla.rename(columns=renombres)

    tabla["Avance (%)"] = (tabla["Devengado"] / tabla["PIM"].replace(0, pd.NA) * 100).round(1)

    total = tabla.drop(columns="Unidad Ejecutora").sum(numeric_only=True)
    total["Unidad Ejecutora"] = "TOTAL"
    pim_total = total.get("PIM", 0)
    total["Avance (%)"] = round(total["Devengado"] / pim_total * 100, 1) if pim_total else float("nan")
    tabla = pd.concat([tabla, pd.DataFrame([total])], ignore_index=True)

    return tabla.sort_values("PIM", ascending=False, ignore_index=True)


## Ejecución

Corre estas dos celdas. Si el filtro de Piura falla, revisa `diagnostico_piura.txt`.

In [ ]:
# ---------------------------------------------------------------------------
# EJECUCIÓN
# ---------------------------------------------------------------------------
df = leer_archivo(RUTA_ARCHIVO)
mostrar_columnas(df)   # revisa esta salida antes de confiar en el resultado


In [ ]:
tabla_final = construir_tabla(df)

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("\nEjecución presupuestal - Gobierno Regional de Piura (cierre agosto 2026)\n")
tabla_final


## Exportar a Excel

In [ ]:
tabla_final.to_excel("ejecucion_piura_agosto_2026.xlsx", index=False)
print("Archivo generado: ejecucion_piura_agosto_2026.xlsx")
